# OPRB Benchmark — 02. GOT-OCR2.0 (0.58B, 2x T4 Data-Parallel)

Zero-shot evaluation of `stepfun-ai/GOT-OCR2_0` (`stepfun-ai/GOT-OCR-2.0-hf`) across 2x T4 GPUs on the OPRB test set.

## 1. Install Dependencies

In [1]:
%pip install -q "tiktoken>=0.7" "verovio>=4.0" "jiwer>=3.0"

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration & Paths (Multi-GPU 2x T4 Enabled)

In [2]:
import concurrent.futures
import csv
import json
import os
import random
import re
import shutil
import threading
import time
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from jiwer import cer, wer
from tqdm.auto import tqdm

NOTEBOOK_START_TIME = time.perf_counter()
MAX_RUNTIME_SECONDS = int(11.2 * 3600)  # Graceful guard before Kaggle's 12h (43,200s) hard timeout

NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
DEVICES = [f"cuda:{i}" for i in range(NUM_GPUS)] if NUM_GPUS > 0 else ["cpu"]
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()} | GPU count: {NUM_GPUS}")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} -> {props.name} ({props.total_memory / 2**30:.1f} GiB)")

MODEL_ID = "stepfun-ai/GOT-OCR2_0"
MODEL_SLUG = "got_ocr_2_0"

CANDIDATE_ROOTS = [
    Path("/kaggle/input/oprb-test-data/oprb_dataset"),
    Path("/kaggle/input/oprb-test-data"),
    Path("./oprb_dataset"),
]
DATA_ROOT = next((p for p in CANDIDATE_ROOTS if (p / "text" / "img").exists()), CANDIDATE_ROOTS[0])
OUTPUT_DIR = Path(f"/kaggle/working/{MODEL_SLUG}_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CATEGORIES = ["text", "nontext"]
MAX_SAMPLES_PER_CATEGORY = None  # None = evaluate all 9,000 test pages
RANDOM_SEED = 42
NORMALIZE_CASE = True

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

PyTorch: 2.11.0+cu130 | CUDA available: True | GPU count: 2
  cuda:0 -> Tesla T4 (14.6 GiB)
  cuda:1 -> Tesla T4 (14.6 GiB)
DATA_ROOT: /kaggle/input/oprb-test-data
OUTPUT_DIR: /kaggle/working/got_ocr_2_0_results
[LOADED] stepfun-ai/GOT-OCR2_0 (stepfun-ai/GOT-OCR-2.0-hf) on cuda:0 in float16
[LOADED] stepfun-ai/GOT-OCR2_0 (stepfun-ai/GOT-OCR-2.0-hf) on cuda:1 in float16
Warm-up worker 0 (cuda:0) done in 31.63s | sample: Figure 16: Lemma 11.2: the dotted curve cannot be part of Γa, b′.
Warm-up worker 1 (cuda:1) done in 26.73s | sample: I e 0-1 is such that for any k E d, (0) k = k. We thus arrive at


## 3. Metadata Parsing & Stratified Test Set Pairing

In [3]:
def parse_occlusion_metadata(stem: str) -> dict:
    """
    Parse OPRB filename convention: OcclusionType__density__batchID...
    Returns occlusion_type, occlusion_density, and batch_id.
    """
    parts = stem.split("__", 2)
    return {
        "occlusion_type": parts[0] if len(parts) > 0 and parts[0] else "unknown",
        "occlusion_density": parts[1] if len(parts) > 1 and parts[1] else "unknown",
        "batch_id": parts[2] if len(parts) > 2 and parts[2] else "unknown",
    }


def collect_test_pairs(categories: list[str]) -> list[dict]:
    pairs = []
    for category in categories:
        image_dir = DATA_ROOT / category / "img"
        gt_dir = DATA_ROOT / category / "gt"
        images = {p.stem: p for p in image_dir.glob("*.jpg")}
        gts = {p.stem: p for p in gt_dir.glob("*.txt")}
        common = sorted(images.keys() & gts.keys())
        print(f"{category}: images={len(images):,}, gt={len(gts):,}, paired={len(common):,}")
        if not common:
            raise RuntimeError(f"No paired files found for category '{category}' under {DATA_ROOT}")
        if MAX_SAMPLES_PER_CATEGORY is not None:
            rng = random.Random(RANDOM_SEED)
            common = sorted(rng.sample(common, min(MAX_SAMPLES_PER_CATEGORY, len(common))))
        for stem in common:
            pairs.append({
                "category": category,
                "sample_id": stem,
                "image_path": str(images[stem]),
                "gt_path": str(gts[stem]),
                **parse_occlusion_metadata(stem),
            })
    # Deterministic shuffle so text/nontext and all occlusion types/densities are interleaved evenly
    random.Random(RANDOM_SEED).shuffle(pairs)
    return pairs


pairs = collect_test_pairs(CATEGORIES)
print(f"Total samples queued for evaluation: {len(pairs):,}")
pd.DataFrame(pairs)[["category", "occlusion_type", "occlusion_density", "batch_id", "sample_id"]].head(5)

text: images=6,000, gt=6,000, paired=6,000
nontext: images=3,000, gt=3,000, paired=3,000
Total samples queued for evaluation: 9,000


## 4. Ground-Truth Reader, Text Normalization & Metrics

In [4]:
def read_gt_tokens(gt_path: str) -> list[str]:
    """Read tab-separated OPRB GT file and extract column 0 (token) in row order."""
    tokens = []
    with open(gt_path, "r", encoding="utf-8-sig", errors="replace") as f:
        for line in f:
            fields = line.rstrip("\r\n").split("\t")
            if fields and fields[0].strip():
                tokens.append(fields[0].strip())
    return tokens


def clean_text(text: str) -> str:
    """Apply Unicode NFKC normalization, whitespace collapsing, and casefolding."""
    text = unicodedata.normalize("NFKC", text or "")
    text = re.sub(r"\s+", " ", text).strip()
    return text.casefold() if NORMALIZE_CASE else text


def bag_of_words_prf(reference: str, prediction: str) -> tuple[float, float, float]:
    """Order-invariant bag-of-words token Precision, Recall, and F1-score."""
    ref = Counter(reference.split())
    hyp = Counter(prediction.split())
    matched = sum((ref & hyp).values())
    ref_n, hyp_n = sum(ref.values()), sum(hyp.values())
    precision = matched / hyp_n if hyp_n else (1.0 if ref_n == 0 else 0.0)
    recall = matched / ref_n if ref_n else (1.0 if hyp_n == 0 else 0.0)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return float(precision), float(recall), float(f1)


def compute_metrics(ground_truth: str, prediction: str) -> dict:
    ref = clean_text(ground_truth)
    hyp = clean_text(prediction)
    if not ref:
        raise ValueError("Empty ground-truth transcript; cannot compute CER/WER.")
    p, r, f1 = bag_of_words_prf(ref, hyp)
    return {
        "cer": float(cer(ref, hyp)),
        "wer": float(wer(ref, hyp)),
        "token_precision": p,
        "token_recall": r,
        "token_f1": f1,
        "gt_words": len(ref.split()),
        "pred_words": len(hyp.split()),
    }


_toy = compute_metrics("The cat sat", "The cat")
assert abs(_toy["wer"] - 1 / 3) < 1e-6, _toy
print("Metric sanity check passed:", _toy)

Metric sanity check passed: {'cer': 0.36363636363636365, 'wer': 0.3333333333333333, 'token_precision': 1.0, 'token_recall': 0.6666666666666666, 'token_f1': 0.8, 'gt_words': 3, 'pred_words': 2}


## 5. Multi-GPU Model Initialization & Warm-Up

In [5]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from transformers.cache_utils import DynamicCache

# Patch DynamicCache for backwards compatibility with any remote modeling code
if not hasattr(DynamicCache, "seen_tokens"):
    DynamicCache.seen_tokens = property(lambda self: self.get_seq_length())
if not hasattr(DynamicCache, "get_max_length"):
    DynamicCache.get_max_length = lambda self: None
if not hasattr(DynamicCache, "get_usable_length"):
    DynamicCache.get_usable_length = lambda self, new_seq_length, layer_idx=0: self.get_seq_length(layer_idx)

HF_GOT_ID = "stepfun-ai/GOT-OCR-2.0-hf"
MAX_NEW_TOKENS = 1024

processor = AutoProcessor.from_pretrained(HF_GOT_ID)

def load_got_replica(device: str):
    m = AutoModelForImageTextToText.from_pretrained(
        HF_GOT_ID,
        torch_dtype=torch.float16,
        device_map={"": device},
    )
    m.eval()
    print(f"[LOADED] {MODEL_ID} ({HF_GOT_ID}) on {device} in float16")
    return m

MODELS = [load_got_replica(dev) for dev in DEVICES]
LOCKS = [threading.Lock() for _ in DEVICES]
NUM_WORKERS = len(MODELS)

@torch.inference_mode()
def predict_page(image_path: str, worker_id: int = 0) -> tuple[str, float, int, bool]:
    device = DEVICES[worker_id]
    model = MODELS[worker_id]
    with LOCKS[worker_id]:
        start = time.perf_counter()
        with Image.open(image_path) as im:
            image = im.convert("RGB")

        inputs = processor(image, return_tensors="pt").to(device)
        # Cast floating point tensors to float16
        for k, v in list(inputs.items()):
            if torch.is_tensor(v) and v.is_floating_point():
                inputs[k] = v.to(dtype=torch.float16)

        output_ids = model.generate(
            **inputs,
            do_sample=False,
            tokenizer=processor.tokenizer,
            stop_strings="<|im_end|>",
            max_new_tokens=MAX_NEW_TOKENS,
            repetition_penalty=1.05,
        )
        new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
        text = processor.decode(new_tokens, skip_special_tokens=True).strip()
        if device.startswith("cuda"):
            torch.cuda.synchronize(device)
        elapsed = time.perf_counter() - start
        n_tokens = int(new_tokens.numel())
        truncated = n_tokens >= MAX_NEW_TOKENS
        del inputs, output_ids, new_tokens
        return text, elapsed, n_tokens, truncated

# Warm-up all GPU replicas
for wid in range(NUM_WORKERS):
    _wt, _wsec, _, _ = predict_page(pairs[wid]["image_path"], worker_id=wid)
    print(f"Warm-up worker {wid} ({DEVICES[wid]}) done in {_wsec:.2f}s | sample: {_wt[:120]}")

[LOADED] stepfun-ai/GOT-OCR2_0 (stepfun-ai/GOT-OCR-2.0-hf) on cuda:0 in float16
[LOADED] stepfun-ai/GOT-OCR2_0 (stepfun-ai/GOT-OCR-2.0-hf) on cuda:1 in float16
Warm-up worker 0 (cuda:0) done in 31.63s | sample: Figure 16: Lemma 11.2: the dotted curve cannot be part of Γa, b′.
Warm-up worker 1 (cuda:1) done in 26.73s | sample: I e 0-1 is such that for any k E d, (0) k = k. We thus arrive at


## 6. Zero-Shot Data-Parallel Inference Loop with Checkpointing

In [6]:
RESULTS_CSV = OUTPUT_DIR / "per_image_results.csv"

FIELDNAMES = [
    "category", "sample_id", "occlusion_type", "occlusion_density", "batch_id",
    "model_id", "status", "error", "prediction", "ground_truth",
    "latency_sec", "generated_tokens", "possibly_truncated",
    "cer", "wer", "token_precision", "token_recall", "token_f1",
    "gt_words", "pred_words",
]

completed = set()
if RESULTS_CSV.exists():
    try:
        prev_df = pd.read_csv(RESULTS_CSV, usecols=["sample_id", "status"])
        completed = set(prev_df.loc[prev_df["status"] == "ok", "sample_id"].astype(str).tolist())
        print(f"Resuming from checkpoint: {len(completed):,} samples already completed.")
    except Exception as e:
        print("Checkpoint read warning:", e)

pending_pairs = [s for s in pairs if s["sample_id"] not in completed]
print(f"Pending samples to evaluate: {len(pending_pairs):,} across {NUM_WORKERS} GPU worker(s)")


def evaluate_single_sample(args: tuple[int, dict]) -> dict:
    idx, sample = args
    worker_id = idx % NUM_WORKERS
    stem = sample["sample_id"]
    row = {
        "category": sample["category"],
        "sample_id": stem,
        "occlusion_type": sample["occlusion_type"],
        "occlusion_density": sample["occlusion_density"],
        "batch_id": sample["batch_id"],
        "model_id": MODEL_ID,
        "status": "error",
        "error": "",
        "prediction": "",
        "ground_truth": "",
        "latency_sec": np.nan,
        "generated_tokens": 0,
        "possibly_truncated": False,
        "cer": np.nan,
        "wer": np.nan,
        "token_precision": np.nan,
        "token_recall": np.nan,
        "token_f1": np.nan,
        "gt_words": 0,
        "pred_words": 0,
    }
    try:
        prediction, elapsed_sec, n_tokens, truncated = predict_page(sample["image_path"], worker_id=worker_id)
        ground_truth = " ".join(read_gt_tokens(sample["gt_path"]))
        metrics = compute_metrics(ground_truth, prediction)
        row.update({
            "status": "ok",
            "prediction": prediction,
            "ground_truth": ground_truth,
            "latency_sec": elapsed_sec,
            "generated_tokens": n_tokens,
            "possibly_truncated": truncated,
            **metrics,
        })
    except Exception as exc:
        row["error"] = repr(exc)
        print(f"[ERROR] {stem} (worker {worker_id}): {exc!r}")
    return row


write_header = not RESULTS_CSV.exists() or RESULTS_CSV.stat().st_size == 0
with open(RESULTS_CSV, "a", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=FIELDNAMES)
    if write_header:
        writer.writeheader()
        csv_file.flush()

    # Process in chunks of 16 so we can check the 11.2h time guard and flush checkpoints cleanly
    CHUNK_SIZE = 16
    pbar = tqdm(total=len(pending_pairs), desc=f"Evaluating {MODEL_SLUG} ({NUM_WORKERS}x GPU)")
    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
        for start_idx in range(0, len(pending_pairs), CHUNK_SIZE):
            if (time.perf_counter() - NOTEBOOK_START_TIME) >= MAX_RUNTIME_SECONDS:
                print(f"[TIME_GUARD] Reached {MAX_RUNTIME_SECONDS/3600:.1f}h session budget. Stopping loop cleanly to save summary & zip.")
                break

            chunk = [
                (start_idx + offset, sample)
                for offset, sample in enumerate(pending_pairs[start_idx : start_idx + CHUNK_SIZE])
            ]
            for row in pool.map(evaluate_single_sample, chunk):
                writer.writerow(row)
                pbar.update(1)
            csv_file.flush()

            if ((start_idx // CHUNK_SIZE) + 1) % 10 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
    pbar.close()

results = pd.read_csv(RESULTS_CSV)
print(f"Completed: {(results['status'] == 'ok').sum():,} / {len(results):,} OK")
results[["category", "occlusion_type", "occlusion_density", "cer", "wer", "token_f1", "latency_sec", "status"]].head(10)

PyTorch: 2.11.0+cu130 | CUDA available: True | GPU count: 2
  cuda:0 -> Tesla T4 (14.6 GiB)
  cuda:1 -> Tesla T4 (14.6 GiB)
DATA_ROOT: /kaggle/input/oprb-test-data
OUTPUT_DIR: /kaggle/working/got_ocr_2_0_results
[LOADED] stepfun-ai/GOT-OCR2_0 (stepfun-ai/GOT-OCR-2.0-hf) on cuda:0 in float16
[LOADED] stepfun-ai/GOT-OCR2_0 (stepfun-ai/GOT-OCR-2.0-hf) on cuda:1 in float16
Warm-up worker 0 (cuda:0) done in 31.63s | sample: Figure 16: Lemma 11.2: the dotted curve cannot be part of Γa, b′.
Warm-up worker 1 (cuda:1) done in 26.73s | sample: I e 0-1 is such that for any k E d, (0) k = k. We thus arrive at


## 7. Macro-Aggregation (Overall, Category, Occlusion Type & Density)

In [7]:
from IPython.display import display

ok = results.loc[results["status"] == "ok"].copy()
assert len(ok) > 0, "No successful predictions found! Check the 'error' column in per_image_results.csv."

metric_cols = ["cer", "wer", "token_precision", "token_recall", "token_f1", "latency_sec"]

def aggregate_metrics(df: pd.DataFrame, group_by: list[str]) -> pd.DataFrame:
    return (
        df.groupby(group_by, dropna=False)
        .agg(
            n=("sample_id", "count"),
            **{f"mean_{c}": (c, "mean") for c in metric_cols},
            truncated=("possibly_truncated", "sum"),
        )
        .reset_index()
    )

overall = {f"mean_{m}": float(ok[m].mean()) for m in metric_cols}
overall.update({
    "model_id": MODEL_ID,
    "num_gpus_used": NUM_WORKERS,
    "n_success": int(len(ok)),
    "n_errors": int(len(results) - len(ok)),
    "n_possibly_truncated": int(ok["possibly_truncated"].sum()),
    "total_wall_time_sec": float(time.perf_counter() - NOTEBOOK_START_TIME),
})

by_category = aggregate_metrics(ok, ["category"])
by_type = aggregate_metrics(ok, ["category", "occlusion_type"])
by_density = aggregate_metrics(ok, ["category", "occlusion_density"])

print("OVERALL MACRO-AVERAGED METRICS:")
print(json.dumps(overall, indent=2))
display(by_category)
display(by_type)
display(by_density)

by_category.to_csv(OUTPUT_DIR / "summary_by_category.csv", index=False)
by_type.to_csv(OUTPUT_DIR / "summary_by_occlusion_type.csv", index=False)
by_density.to_csv(OUTPUT_DIR / "summary_by_density.csv", index=False)
(OUTPUT_DIR / "run_config.json").write_text(json.dumps(overall, indent=2), encoding="utf-8")

PyTorch: 2.11.0+cu130 | CUDA available: True | GPU count: 2
  cuda:0 -> Tesla T4 (14.6 GiB)
  cuda:1 -> Tesla T4 (14.6 GiB)
DATA_ROOT: /kaggle/input/oprb-test-data
OUTPUT_DIR: /kaggle/working/got_ocr_2_0_results
[LOADED] stepfun-ai/GOT-OCR2_0 (stepfun-ai/GOT-OCR-2.0-hf) on cuda:0 in float16
[LOADED] stepfun-ai/GOT-OCR2_0 (stepfun-ai/GOT-OCR-2.0-hf) on cuda:1 in float16
Warm-up worker 0 (cuda:0) done in 31.63s | sample: Figure 16: Lemma 11.2: the dotted curve cannot be part of Γa, b′.
Warm-up worker 1 (cuda:1) done in 26.73s | sample: I e 0-1 is such that for any k E d, (0) k = k. We thus arrive at


## 8. Output Validation & Packaging (`results.zip`)

In [8]:
expected_files = [
    "per_image_results.csv",
    "summary_by_category.csv",
    "summary_by_occlusion_type.csv",
    "summary_by_density.csv",
    "run_config.json",
]
for fname in expected_files:
    fpath = OUTPUT_DIR / fname
    assert fpath.exists() and fpath.stat().st_size > 0, f"Missing or empty output: {fpath}"
    print(f"[OK] {fname} ({fpath.stat().st_size:,} bytes)")

zip_path = shutil.make_archive(f"/kaggle/working/{MODEL_SLUG}_results", "zip", root_dir=OUTPUT_DIR)
print("Packaged results archive:", zip_path)

[OK] per_image_results.csv (14,311,794 bytes)
[OK] summary_by_category.csv (367 bytes)
[OK] summary_by_occlusion_type.csv (1,795 bytes)
[OK] summary_by_density.csv (116,327 bytes)
[OK] run_config.json (412 bytes)
Packaged results archive: /kaggle/working/got_ocr_2_0_results.zip
